# Encoder Fine-tuning — bge-small-en-v1.5 (Exp_2)

Fine-tunea el encoder con **TripletLoss** sobre triplets a nivel de **chunk** (no doc completo).

**Fix v2**: usamos chunks como positivos/negativos en lugar de documentos completos.
Esto alinea el entrenamiento con la inferencia (FAISS busca sobre chunks, no sobre docs).

**Corre en Colab T4 — ~15 min.**

Archivos necesarios en Drive:
- `retrieval_index.json`
- `Data/consultas_centro_control.json`

Output: `encoder_finetuned/` guardado en Drive.

In [1]:
# ── CELL 0: Config ──────────────────────────────────────────────────────────
DRIVE_PATH   = "/content/drive/MyDrive/MASTER/Tercer_semestre/NLP_2/Competencia/Exp_2"
ENCODER_BASE = "BAAI/bge-small-en-v1.5"
ENCODER_OUT  = "encoder_finetuned"   # directorio local en Colab → se sube a Drive al final

In [2]:
# ── CELL 1: Mount Drive + Install ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run([
    "pip", "install", "-q",
    "sentence-transformers>=3.0.0",
    "datasets>=2.20.0",
], check=True)

print("\n=== IMPORTANTE ===")
print("Reinicia el runtime (Runtime → Restart runtime) y continúa desde la siguiente celda.")

Mounted at /content/drive

=== IMPORTANTE ===
Reinicia el runtime (Runtime → Restart runtime) y continúa desde la siguiente celda.


In [3]:
# ── CELL 2: Imports (después del restart) ───────────────────────────────────
import os, json, shutil
import torch
from pathlib import Path
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import TripletLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU: NVIDIA A100-SXM4-40GB


/tmp/ipykernel_689/1061937707.py:7: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import TripletLoss
/tmp/ipykernel_689/1061937707.py:8: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments
/tmp/ipykernel_689/1061937707.py:9: DeprecationWarning: Importing from 'sentence_transformers.trainer' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.trainer' instead.
  from sentence_transformers.trainer import SentenceTransformerTrainer


In [ ]:
# ── CELL 3: Copiar archivos desde Drive ─────────────────────────────────────
os.makedirs("Data", exist_ok=True)

for src_rel, dst in [
    ("retrieval_index.json",               "retrieval_index.json"),
    ("Data/consultas_centro_control.json", "Data/consultas_centro_control.json"),
]:
    if not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        shutil.copy(os.path.join(DRIVE_PATH, src_rel), dst)
        print(f"Copiado: {src_rel}")

print("Archivos listos.")

In [ ]:
# ── CELL 4: Cargar chunks agrupados por doc_id ───────────────────────────────
# Usamos chunks (no docs completos) como positivos/negativos.
# FAISS busca sobre chunks → entrenamiento debe usar la misma distribución.
from collections import defaultdict

with open("retrieval_index.json") as f:
    chunks = json.load(f)

chunks_by_doc = defaultdict(list)
for c in chunks:
    chunks_by_doc[c["doc_id"]].append(c["text"])

print(f"Chunks totales: {len(chunks)}")
print(f"Docs con chunks: {len(chunks_by_doc)}")
print(f"Chunks/doc promedio: {len(chunks)/len(chunks_by_doc):.1f}")

In [ ]:
# ── CELL 5: Construir triplets a nivel de chunk ──────────────────────────────
# Por cada consulta: una tripla por cada chunk del doc correcto,
# emparejado con el primer chunk del doc incorrecto.
# 622 consultas × ~14 chunks/doc ≈ ~8,700 triplets.
with open("Data/consultas_centro_control.json") as f:
    consultas = json.load(f)

triplet_data = {"anchor": [], "positive": [], "negative": []}
skipped = 0

for item in consultas:
    if "hard_negative_doc_id" not in item:
        continue
    pos_chunks = chunks_by_doc.get(item["doc_id"], [])
    neg_chunks = chunks_by_doc.get(item["hard_negative_doc_id"], [])
    if not pos_chunks or not neg_chunks:
        skipped += 1
        continue
    neg_sample = neg_chunks[0]
    for pos_text in pos_chunks:
        triplet_data["anchor"].append(item["query"])
        triplet_data["positive"].append(pos_text)
        triplet_data["negative"].append(neg_sample)

train_dataset = Dataset.from_dict(triplet_data)
print(f"Triplets: {len(train_dataset)} (saltados: {skipped})")
print(f"(v1 con docs completos: 622 — v2 con chunks: {len(train_dataset)})")

In [ ]:
# ── CELL 6: Fine-tuning con TripletLoss ─────────────────────────────────────
model_ft = SentenceTransformer(ENCODER_BASE)
loss     = TripletLoss(model=model_ft)

train_args = SentenceTransformerTrainingArguments(
    output_dir=ENCODER_OUT,
    num_train_epochs=3,       # menos épocas → menos overfitting (era 5)
    per_device_train_batch_size=32,
    learning_rate=5e-6,       # más conservador (era 2e-5)
    warmup_steps=0.1,         # ratio como float (nuevo API de transformers v5)
    fp16=True,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=20,
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model_ft,
    args=train_args,
    train_dataset=train_dataset,
    loss=loss,
)

trainer.train()
model_ft.save(ENCODER_OUT)
print(f"Encoder guardado en: {ENCODER_OUT}/")

In [8]:
# ── CELL 7: Subir encoder fine-tuned a Drive ─────────────────────────────────
encoder_dst = os.path.join(DRIVE_PATH, ENCODER_OUT)
if os.path.exists(encoder_dst):
    shutil.rmtree(encoder_dst)
shutil.copytree(ENCODER_OUT, encoder_dst)
print(f"Guardado en Drive: {encoder_dst}/")
print("\nListo. Ahora corre rag.ipynb en Colab para reconstruir el índice con el encoder fine-tuned.")

Guardado en Drive: /content/drive/MyDrive/MASTER/Tercer_semestre/NLP_2/Competencia/Exp_2/encoder_finetuned/

Listo. Ahora corre rag.ipynb en Colab para reconstruir el índice con el encoder fine-tuned.
